# Outlook subscription forecasting: Part 2
This notebook reproduces the analysis using the supplied workbook. Values are subscription balances and expiries, not sales or revenue. Install `pandas numpy scipy statsmodels matplotlib openpyxl` to run. Put this notebook beside the source workbook. No calendar dates are inferred for weekly or fortnightly issues.

In [ ]:
# Import the analysis tools and set reproducible paths and randomness.
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.exponential_smoothing.ets import ETSModel
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
np.random.seed(42)
source = Path('Data for sales forecasting and trend line analysis (1).xlsx')
out = Path('part2_results'); out.mkdir(exist_ok=True)

## Read and clean the source
The sheet is arranged in three year blocks. We convert comma-separated values to numbers and apply the user's confirmed correction of 146,222. Monthly records get calendar month dates. Issue records retain year and issue number, because actual issue dates were not supplied.

In [ ]:
# Reshape each year block into a consistent long table.
book = pd.ExcelFile(source)
records = []
for sheet in book.sheet_names:
    raw = pd.read_excel(source, sheet_name=sheet, header=None)
    monthly = sheet in ['Outlook Business.', 'Outlook Traveller', 'Outlook Money']
    for offset, year in [(0,2020),(4,2021),(8,2022)]:
        for _, row in raw.iloc[2:, offset:offset+3].iterrows():
            period, opening, expiry = row.iloc[0], row.iloc[1], row.iloc[2]
            if pd.isna(period): continue
            if sheet == 'Outlook Hindi' and year == 2020 and int(period) == 25: opening = 146222
            if isinstance(opening, str): opening = int(opening.replace(',', ''))
            records.append([sheet.rstrip('.'), year, int(period), 'monthly' if monthly else 'issue', int(opening), int(expiry)])
data = pd.DataFrame(records, columns=['title','year','period','frequency','opening','expiry'])
data['date'] = pd.to_datetime(dict(year=data.year, month=data.period.where(data.frequency.eq('monthly'), 1), day=1)).where(data.frequency.eq('monthly'))
data.to_csv(out/'cleaned_data.csv', index=False)
print('Rows by title:', data.groupby('title').size().to_dict())
print('Missing values in original fields:', data[['title','year','period','opening','expiry']].isna().sum().to_dict())
print('Duplicates:', data.duplicated(['title','year','period']).sum())

## Descriptive statistics
Standard deviation uses the sample formula. CV is standard deviation divided by the mean. These describe all issue observations; different frequencies and stock versus flow measures must not be added together.

In [ ]:
# Describe opening balances and expiry counts for each title at source frequency.
stats = data.groupby('title')[['opening','expiry']].agg(['count','mean','median','std','min','max'])
for measure in ['opening','expiry']: stats[(measure,'CV')] = stats[(measure,'std')] / stats[(measure,'mean')]
stats.to_csv(out/'descriptive_statistics.csv')
print(stats.round(2).to_string())

## Monthly analysis: three monthly titles
The combined series below covers only the three monthly titles. Opening balances are added across titles at the same month start; they are not summed across months. Expiries are period flows. Year-over-year and month-over-month growth compare like measures. ADF tests the null hypothesis of a unit root; a p-value above 0.05 does not prove non-stationarity.

In [ ]:
# Build same-frequency, same-date series and calculate growth.
monthly = data[data.frequency.eq('monthly')].copy()
monthly['series'] = monthly.title
overall = monthly.groupby('date', as_index=False)[['opening','expiry']].sum()
overall['series'] = 'Combined monthly titles'
seriesdata = pd.concat([monthly[['date','series','opening','expiry']], overall], ignore_index=True).sort_values(['series','date'])
seriesdata['mom_opening'] = seriesdata.groupby('series').opening.pct_change(fill_method=None)
seriesdata['yoy_opening'] = seriesdata.groupby('series').opening.pct_change(12, fill_method=None)
seriesdata.to_csv(out/'monthly_growth.csv', index=False)
print('Growth and first/last balances:')
for name, g in seriesdata.groupby('series'):
    g=g.sort_values('date'); start,end=g.opening.iloc[0],g.opening.iloc[-1]
    cagr=(end/start)**(12/35)-1
    print(f'{name}: Jan 2020 {start:,.0f}; Dec 2022 {end:,.0f}; total change {end/start-1:.2%}; annualized over 35 months {cagr:.2%}; last YoY {g.yoy_opening.iloc[-1]:.2%}')
    print('  highest',g.loc[g.opening.idxmax(),['date','opening']].to_dict(),'lowest',g.loc[g.opening.idxmin(),['date','opening']].to_dict())
    print('  ADF p-value',round(adfuller(g.opening, autolag='AIC')[1],4))
    print('  mean, median, std, min, max, CV',*[round(v,2) for v in [g.opening.mean(),g.opening.median(),g.opening.std(),g.opening.min(),g.opening.max(),g.opening.std()/g.opening.mean()]])
fig,ax=plt.subplots(figsize=(10,5))
for name,g in monthly.groupby('series'): ax.plot(g.date,g.opening,label=name)
ax.set(title='Monthly opening active subscriptions',ylabel='Active subscriptions',xlabel='Month'); ax.legend(); fig.tight_layout(); fig.savefig(out/'monthly_balances.png',dpi=160); plt.show()

## Seasonality
Classical additive decomposition uses a 12-month cycle. A seasonal index compares each calendar month's average with that series' overall average. Neither establishes a repeatable seasonal pattern with only three years, especially with unexplained shifts. Index values above 1 mean the month averaged above the overall mean, not proof of a causal seasonal effect.

In [ ]:
# Examine monthly profiles and perform a descriptive decomposition.
seasonal_rows=[]
for name,g in seriesdata.groupby('series'):
    s=g.sort_values('date').set_index('date').opening.asfreq('MS')
    dec=seasonal_decompose(s, model='additive',period=12)
    idx=s.groupby(s.index.month).mean()/s.mean()
    seasonal_rows.extend([{'series':name,'month':m,'index':v,'additive_seasonal_component':dec.seasonal[dec.seasonal.index.month==m].iloc[0]} for m,v in idx.items()])
    print(name,'largest raw month index',int(idx.idxmax()),round(idx.max(),3),'smallest',int(idx.idxmin()),round(idx.min(),3))
pd.DataFrame(seasonal_rows).to_csv(out/'seasonal_indices.csv',index=False)

## Forecast comparison
Hold out July–December 2022. Forecast all six months from the end of June without using held-out observations. For the 3-month moving average, feed forecasts back into the rolling window. Select the lowest holdout RMSE; MAPE is a complementary percentage measure (undefined for zero actuals). ETS means an error-trend-seasonal model. We compare simple models and only fit an additive seasonal model where the 30-month training sample permits it. SARIMA is omitted: two and a half training cycles and unexplained jumps give little support for a more parameter-heavy seasonal model. Forecasts start in January 2023, immediately after the data, and are not forecasts for today.

In [ ]:
# Define fixed-origin forecasting models. ETS models use additive errors.
def predict(train, model, steps):
    x=np.asarray(train,dtype=float)
    if model=='Naive': return np.repeat(x[-1],steps)
    if model=='Seasonal naive': return np.resize(x[-12:],steps)
    if model=='Moving average (3)':
        hist=list(x); result=[]
        for _ in range(steps):
            v=np.mean(hist[-3:]); result.append(v); hist.append(v)
        return np.array(result)
    spec={'SES':(None,None), 'Holt':('add',None), 'Holt-Winters':('add','add')}[model]
    fit=ETSModel(x,error='add',trend=spec[0],seasonal=spec[1],seasonal_periods=12 if spec[1] else None,initialization_method='estimated').fit(disp=False, maxiter=2000)
    return np.asarray(fit.forecast(steps))
models=['Naive','Seasonal naive','Moving average (3)','SES','Holt','Holt-Winters']
rows=[]
for name,g in seriesdata.groupby('series'):
    for measure in ['opening','expiry']:
        x=g.sort_values('date')[measure].to_numpy(); train,test=x[:-6],x[-6:]
        for model in models:
            try:
                pred=predict(train,model,6)
                rows.append({'series':name,'measure':measure,'model':model,'RMSE':np.sqrt(np.mean((test-pred)**2)),'MAPE_pct':np.mean(np.abs((test-pred)/test))*100 if np.all(test!=0) else np.nan})
            except Exception as e: print('Not fitted',name,measure,model,str(e))
comparison=pd.DataFrame(rows).sort_values(['series','measure','RMSE'])
comparison.to_csv(out/'model_comparison.csv',index=False)
print(comparison.round(2).to_string(index=False))

## Prediction intervals and refit
After model selection, refit the selected model to all 36 observations. For ETS models we simulate 5,000 paths using fitted model errors. For the moving average, we recursively simulate paths by resampling centred historical one-step residuals. The 2.5th and 97.5th percentiles provide approximate 95% prediction intervals conditional on the model. These intervals do not capture all uncertainty from model selection, questionable source values or structural change. The lower bound is clipped to zero because counts cannot be negative; the raw bound is retained.

In [ ]:
# Refit the selected model and simulate predictive paths.
forecast_rows=[]
rng=np.random.default_rng(42)
for (name,measure),group in comparison.groupby(['series','measure']):
    best=group.iloc[0].model
    x=seriesdata[seriesdata.series.eq(name)].sort_values('date')[measure].to_numpy()
    pred=predict(x,best,12)
    if best in ['SES','Holt','Holt-Winters']:
        spec={'SES':(None,None),'Holt':('add',None),'Holt-Winters':('add','add')}[best]
        fit=ETSModel(x,error='add',trend=spec[0],seasonal=spec[1],seasonal_periods=12 if spec[1] else None,initialization_method='estimated').fit(disp=False,maxiter=2000)
        paths=np.asarray(fit.simulate(nsimulations=12,anchor='end',repetitions=5000,rng=rng))
        lo,hi=np.quantile(paths,[.025,.975],axis=1)
    elif best=='Moving average (3)':
        residuals=np.array([x[i]-x[i-3:i].mean() for i in range(3,len(x))]); residuals-=residuals.mean()
        histories=np.tile(x[-3:],(5000,1)); paths=[]
        for h in range(12):
            values=histories[:,-3:].mean(axis=1)+rng.choice(residuals,size=5000)
            paths.append(values); histories=np.column_stack((histories,values))
        lo,hi=np.quantile(np.array(paths),[.025,.975],axis=1)
    else:
        raise ValueError('Add simulation logic for selected baseline: '+best)
    for date,p,l,h in zip(pd.date_range('2023-01-01',periods=12,freq='MS'),pred,lo,hi):
        forecast_rows.append({'series':name,'measure':measure,'date':date,'selected_model':best,'forecast':p,'lower_raw':l,'lower_nonnegative':max(0,l),'upper':h})
forecast=pd.DataFrame(forecast_rows)
forecast.to_csv(out/'forecasts_2023.csv',index=False)
print(forecast[forecast.date.eq('2023-01-01')].round(1).to_string(index=False))

## Interpretation and limitations
- The combined monthly titles exclude Outlook Hindi and Outlook India. They cannot be included in monthly totals without issue dates and an agreed month-end or month-start convention.
- A six-month holdout is one short evaluation window. Model ranking can change in other windows. Intervals are conditional on the chosen model and are not a validated 95% coverage claim.
- Large unexplained values and year-boundary changes are retained as supplied. The user's correction is explicitly documented above.
- Three annual cycles offer weak evidence for seasonality and do not justify assuming stable patterns.
- Stock balances cannot be summed across time or interpreted as new sales. Expiries are counts reaching expiry, not necessarily cancellations or lost subscribers.